# NLI fine-tune data preparation (ContraDoc, held-out balanced sample as test)

Builds train / val / test sentence-pair files for the binary contradiction NLI fine-tune handoff.

**Pool design:**
- **Test = balanced 200** (the canonical sample built in notebook 01).
- **Train + Val = the remaining usable YES docs whose `base_doc_id` does not appear in balanced 200.**

This means no base story (e.g. `3499318673` and its `_1, _3, _5, ...` variants) is shared between train and test pools, so train pool docs never share text with test pool docs. The full balanced 200 stays intact as the downstream pipeline eval set.

**Output:** `data/processed/ContraDoc/nli/{nli_train.csv, nli_val.csv, nli_test.csv, nli_splits.json}`

**Schema (per-row):**

| column | meaning |
|---|---|
| `pair_id` | unique id within the split |
| `doc_id` | source ContraDoc doc_id |
| `base_doc_id` | source story prefix (group key) |
| `contra_type` | pipe-separated primitives copied from ContraDoc |
| `premise` / `hypothesis` | verbatim sentences from `text` |
| `label` | 1 = contradiction (gold pair), 0 = not contradiction (in-doc random pair) |
| `kind` | `gold` for positives, `in_doc_negative` for negatives |


## Data leakage prevention strategy

The KG persists every triple from every doc. Layers 1 (structural) and 2 (vector) of the retrieval pipeline are parameterless lookups - they cannot memorize anything. The leakage exposure is entirely at layer 3 (NLI), which is a learned model.

**Two layers of protection are baked into this notebook:**

1. **Held-out test set.** Train / val pool consists of usable YES docs whose `base_doc_id` is NOT in the balanced 200. The entire balanced 200 - including every variant of every base it touches - is reserved for testing. This is enforced by an `assert` on disjoint base_doc_id sets.

2. **Within-doc pair construction.** A positive pair is `(evidence, ref)` from the SAME doc. A negative pair is two non-gold sentences from the SAME doc. We never construct cross-doc pairs, so no example carries text from another doc, let alone another split.

**Downstream pipeline eval is also clean:** because the balanced 200 is fully held out, you can run notebooks 04 / 05 / 06 on every balanced doc without `nli_test_doc_ids` filtering. Every retrieval candidate at test time comes from a doc whose base story the NLI never saw.

**Train-pool composition note:** because the balanced sample greedy-picked rare primitives (Causal, Relation, Emotion, Factual), the leftover train pool is dominated by Content / Numeric / Negation / Perspective. The friend should warm-start from a 3-class NLI checkpoint (e.g. `cross-encoder/nli-deberta-v3-base`) so the encoder already knows the rare phenomena from generic NLI pre-training; the fine-tune just adapts the contradiction head.


In [1]:
import json
import re
from collections import Counter
from difflib import SequenceMatcher
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit


In [2]:
N_TRAIN_FRAC = 0.80  # within the train+val (leftover) pool; rest goes to val
NEG_PER_POS = 4      # in-doc random negatives per gold positive pair
SEED = 42

DATA_DIR = Path("data/processed/ContraDoc")
INPUT_CSV = DATA_DIR / "ContraDoc.csv"
BALANCED_PATH = DATA_DIR / "balanced_sample.json"
FINDABILITY_PATH = DATA_DIR / "findability.json"
OUT_DIR = DATA_DIR / "nli"
OUT_DIR.mkdir(parents=True, exist_ok=True)


## Load inputs

In [3]:
df = pd.read_csv(INPUT_CSV)
balanced = json.loads(BALANCED_PATH.read_text(encoding="utf-8"))
findability = json.loads(FINDABILITY_PATH.read_text(encoding="utf-8"))

balanced_doc_ids = set(balanced["doc_ids"])
print(f"balanced 200 size: {len(balanced_doc_ids)}")

# Build doc_id -> base_doc_id map.
doc_to_base: dict[str, str] = {}
for base, variants in findability["yes_base_doc_groups"].items():
    for did in variants:
        doc_to_base[did] = base

# usable YES pool from findability + drop docs containing 'Other' primitive.
usable_doc_ids = set(findability["usable_doc_ids"])
yes_usable = df[df["id"].astype(str).isin(usable_doc_ids)].copy()
yes_usable["base_doc_id"] = yes_usable["id"].astype(str).map(doc_to_base)
# Match the EDA filter: drop Other-tagged docs.
EXCLUDED = frozenset({"Other"})
yes_usable = yes_usable[
    ~yes_usable["contra_type"].fillna("").map(
        lambda s: bool(set(t.strip() for t in str(s).split("|") if t.strip()) & EXCLUDED)
    )
].reset_index(drop=True)
print(f"usable YES pool (after dropping {sorted(EXCLUDED)}): {len(yes_usable)}")
print(f"  unique base_doc_ids: {yes_usable['base_doc_id'].nunique()}")


balanced 200 size: 150
usable YES pool (after dropping ['Other']): 301
  unique base_doc_ids: 169


## Build train+val and test pools

In [4]:
# bases touched by the balanced 200 - everything with these bases is reserved for TEST.
balanced_bases = set(yes_usable.loc[yes_usable["id"].astype(str).isin(balanced_doc_ids), "base_doc_id"])
print(f"balanced 200 covers {len(balanced_bases)} unique bases")

# TEST = balanced 200 exactly (one row per balanced doc_id).
test_df = yes_usable[yes_usable["id"].astype(str).isin(balanced_doc_ids)].copy()
test_df["split"] = "test"

# TRAIN+VAL pool = usable YES docs whose base is NOT in balanced_bases.
trainval_pool = yes_usable[~yes_usable["base_doc_id"].isin(balanced_bases)].copy()
print(f"train+val candidate pool: {len(trainval_pool)} docs ({trainval_pool['base_doc_id'].nunique()} bases)")
print()

# How many leftover docs share a base with balanced and therefore got dropped from train+val?
leftover = yes_usable[~yes_usable["id"].astype(str).isin(balanced_doc_ids)]
dropped_for_overlap = leftover[leftover["base_doc_id"].isin(balanced_bases)]
print(f"leftover docs dropped to prevent base overlap with test: {len(dropped_for_overlap)}")
print(f"  these are variants of balanced bases - including them in train would leak text into test")


balanced 200 covers 106 unique bases
train+val candidate pool: 74 docs (63 bases)

leftover docs dropped to prevent base overlap with test: 77
  these are variants of balanced bases - including them in train would leak text into test


## Sentence-level helpers

In [5]:
_SENT_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")


def _normalize(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip().lower().rstrip(".!?\"'")


def split_sentences(text: str) -> list[str]:
    return [s for s in _SENT_SPLIT_RE.split(text) if s.strip()]


def find_sentence_idx(target: str, sentences: list[str], threshold: float = 0.85) -> int:
    """Return 0-based sentence index whose normalized text best matches target,
    or -1 if no match meets threshold."""
    t_norm = _normalize(target)
    for i, s in enumerate(sentences):
        s_n = _normalize(s)
        if t_norm and (s_n == t_norm or t_norm in s_n):
            return i
    best_idx, best_score = -1, 0.0
    for i, s in enumerate(sentences):
        score = SequenceMatcher(None, _normalize(s), t_norm).ratio()
        if score > best_score:
            best_idx, best_score = i, score
    return best_idx if best_score >= threshold else -1


## Group-aware train/val split inside the leftover pool

In [6]:
# Group split inside the train+val pool so variants of one base don't cross train/val.
groups_arr = trainval_pool["base_doc_id"].to_numpy()
gss = GroupShuffleSplit(n_splits=1, test_size=1 - N_TRAIN_FRAC, random_state=SEED)
train_local, val_local = next(gss.split(trainval_pool, groups=groups_arr))

train_df = trainval_pool.iloc[train_local].copy()
train_df["split"] = "train"
val_df = trainval_pool.iloc[val_local].copy()
val_df["split"] = "val"

split_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

print("doc-level split:")
print(split_df["split"].value_counts().to_string())
print()

# LEAKAGE ASSERTION: train, val, test base sets are pairwise disjoint.
train_bases = set(train_df["base_doc_id"])
val_bases = set(val_df["base_doc_id"])
test_bases = set(test_df["base_doc_id"])
assert train_bases.isdisjoint(test_bases), f"LEAKAGE: train ∩ test bases = {train_bases & test_bases}"
assert val_bases.isdisjoint(test_bases), f"LEAKAGE: val ∩ test bases = {val_bases & test_bases}"
assert train_bases.isdisjoint(val_bases), f"LEAKAGE: train ∩ val bases = {train_bases & val_bases}"
print(f"leakage check: pairwise base-disjoint OK")
print(f"  train bases: {len(train_bases)}")
print(f"  val bases:   {len(val_bases)}")
print(f"  test bases:  {len(test_bases)}")


doc-level split:
split
test     150
train     58
val       16

leakage check: pairwise base-disjoint OK
  train bases: 50
  val bases:   13
  test bases:  106


## Build sentence-pair examples

In [7]:
def make_pairs_for_doc(row: pd.Series, neg_per_pos: int, rng: np.random.Generator) -> list[dict]:
    text = row["text"]
    sentences = split_sentences(text)
    n = len(sentences)
    if n < 2:
        return []

    evidence = row["evidence"]
    refs = [r for r in str(row["ref_sentences"]).split("|") if r.strip()]
    e_idx = find_sentence_idx(evidence, sentences)
    r_idxs = [find_sentence_idx(r, sentences) for r in refs]
    r_idxs = [i for i in r_idxs if i >= 0]
    if e_idx < 0 or not r_idxs:
        return []

    pairs: list[dict] = []
    excluded: set[tuple[int, int]] = set()

    for r_idx in r_idxs:
        if r_idx == e_idx:
            continue
        a, b = (e_idx, r_idx) if e_idx < r_idx else (r_idx, e_idx)
        excluded.add((a, b))
        pairs.append({
            "premise": sentences[e_idx],
            "hypothesis": sentences[r_idx],
            "label": 1,
            "kind": "gold",
            "premise_idx": e_idx,
            "hypothesis_idx": r_idx,
        })

    all_pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]
    neg_pool = [p for p in all_pairs if p not in excluded]
    n_neg_target = min(neg_per_pos * len(r_idxs), len(neg_pool))
    if n_neg_target == 0:
        return pairs

    chosen_pos = rng.choice(len(neg_pool), size=n_neg_target, replace=False)
    for ci in chosen_pos:
        i, j = neg_pool[int(ci)]
        if rng.random() < 0.5:
            p_idx, h_idx = i, j
        else:
            p_idx, h_idx = j, i
        pairs.append({
            "premise": sentences[p_idx],
            "hypothesis": sentences[h_idx],
            "label": 0,
            "kind": "in_doc_negative",
            "premise_idx": p_idx,
            "hypothesis_idx": h_idx,
        })
    return pairs


rng = np.random.default_rng(SEED)
records: list[dict] = []
skipped: list[str] = []
pair_id = 0
for _, row in split_df.iterrows():
    pairs = make_pairs_for_doc(row, NEG_PER_POS, rng)
    if not pairs:
        skipped.append(str(row["id"]))
        continue
    for p in pairs:
        records.append({
            "pair_id": pair_id,
            "doc_id": str(row["id"]),
            "base_doc_id": row["base_doc_id"],
            "split": row["split"],
            "contra_type": row["contra_type"],
            **p,
        })
        pair_id += 1

pairs_df = pd.DataFrame(records)
print(f"generated {len(pairs_df)} pairs from {split_df['id'].nunique() - len(skipped)} docs")
if skipped:
    print(f"  skipped {len(skipped)} docs (couldn't sentence-locate evidence/refs); first 5: {skipped[:5]}")
print()
print("per-split, per-label counts:")
print(pairs_df.groupby(["split", "label"]).size().unstack(fill_value=0).to_string())


generated 979 pairs from 196 docs
  skipped 28 docs (couldn't sentence-locate evidence/refs); first 5: ['3488771840_10', '3488771888_8', '3489738240_7', '3489738283_6', '3489825772_9']

per-split, per-label counts:
label    0    1
split          
test   564  103
train  208   39
val     56    9


## Persist CSVs and split metadata

In [8]:
for split_name in ("train", "val", "test"):
    sub = pairs_df[pairs_df["split"] == split_name].drop(columns=["split"])
    out_path = OUT_DIR / f"nli_{split_name}.csv"
    sub.to_csv(out_path, index=False)
    n_pos = int((sub["label"] == 1).sum())
    n_neg = int((sub["label"] == 0).sum())
    print(f"  {split_name}: {len(sub):>5} pairs (pos={n_pos}, neg={n_neg})  -> {out_path}")

split_payload = {
    "seed": SEED,
    "design": "held_out_balanced",
    "description": (
        "Test = balanced 200 (exactly). Train+val = usable YES docs whose base_doc_id "
        "is NOT in balanced 200. Train/val split inside the leftover pool by GroupShuffleSplit on base_doc_id."
    ),
    "train_frac_within_leftover": N_TRAIN_FRAC,
    "neg_per_pos": NEG_PER_POS,
    "group_key": "base_doc_id",
    "leakage_strategy": (
        "Train, val, test base_doc_id sets are pairwise disjoint (asserted at runtime). "
        "Pairs are within-doc only. The full balanced 200 is held out as test, so "
        "downstream notebooks 04 / 05 / 06 can be evaluated on the whole balanced sample "
        "without per-doc filtering."
    ),
    "doc_id_split": {
        s: sorted(split_df[split_df["split"] == s]["id"].astype(str).tolist())
        for s in ("train", "val", "test")
    },
    "base_doc_id_split": {
        s: sorted(split_df[split_df["split"] == s]["base_doc_id"].unique().tolist())
        for s in ("train", "val", "test")
    },
    "pair_counts": {
        s: {
            "pos": int(((pairs_df["split"] == s) & (pairs_df["label"] == 1)).sum()),
            "neg": int(((pairs_df["split"] == s) & (pairs_df["label"] == 0)).sum()),
            "total": int((pairs_df["split"] == s).sum()),
        }
        for s in ("train", "val", "test")
    },
    "skipped_doc_ids": skipped,
}
(OUT_DIR / "nli_splits.json").write_text(
    json.dumps(split_payload, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(f"\nsaved metadata: {OUT_DIR / 'nli_splits.json'}")


  train:   247 pairs (pos=39, neg=208)  -> data\processed\ContraDoc\nli\nli_train.csv
  val:    65 pairs (pos=9, neg=56)  -> data\processed\ContraDoc\nli\nli_val.csv
  test:   667 pairs (pos=103, neg=564)  -> data\processed\ContraDoc\nli\nli_test.csv

saved metadata: data\processed\ContraDoc\nli\nli_splits.json


## Peek a few examples

In [9]:
# Eyeball a few train and test examples.
train_pos = pairs_df[(pairs_df["split"] == "train") & (pairs_df["label"] == 1)].head(2)
test_pos  = pairs_df[(pairs_df["split"] == "test")  & (pairs_df["label"] == 1)].head(2)

print("=== TRAIN positive examples ===")
for _, r in train_pos.iterrows():
    print(f"[{r['doc_id']:<14} {r['contra_type']}]")
    print(f"  premise   : {r['premise'][:140]}")
    print(f"  hypothesis: {r['hypothesis'][:140]}")
    print()

print("=== TEST positive examples ===")
for _, r in test_pos.iterrows():
    print(f"[{r['doc_id']:<14} {r['contra_type']}]")
    print(f"  premise   : {r['premise'][:140]}")
    print(f"  hypothesis: {r['hypothesis'][:140]}")
    print()


=== TRAIN positive examples ===
[3488771845_4   Content]
  premise   : He meets Co-Tan, a member of the highest human race of mainland Caspak, the Galu, fully human and of a neolithic cultural level.They enter t
  hypothesis: Finally, though, they are discovered by Wieroo.

[3488771868_7   Emotion/Mood/Feeling]
  premise   : Various ominous happenings begin to take place at Bartram-Haugh; it becomes increasingly difficult for Maud and Millicent to find any route 
  hypothesis: During her stay, Maud is subject to various attempts by Dudley to court her, but she rejects him thoroughly on each occasion.

=== TEST positive examples ===
[3488771854_6   Content|Emotion/Mood/Feeling]
  premise   : Sylvia celebrates her brother Owen's death.
  hypothesis: Sylvia leaves her father's house to mourn her brother Owen's death.

[3488771854_8   Relation]
  premise   : There Brazen and Plume compete to recruit 'Wilful', unaware of 'his' real identity.Kite abducts 'him' for Brazen while Plume duels wi

## Hand-off note for the fine-tune task

`nli_train.csv`, `nli_val.csv`, `nli_test.csv` are ready to consume. The pool design holds out the entire balanced 200 sample as test, so the friend's standalone NLI metric on `nli_test.csv` is the same population that downstream pipeline notebooks (04 / 05 / 06) will evaluate end-to-end.

1. **Load via HuggingFace `datasets`:**
   ```python
   from datasets import load_dataset
   ds = load_dataset("csv", data_files={
       "train": "data/processed/ContraDoc/nli/nli_train.csv",
       "val":   "data/processed/ContraDoc/nli/nli_val.csv",
       "test":  "data/processed/ContraDoc/nli/nli_test.csv",
   })
   ```

2. **Architecture:** 2-class head (binary contradiction). **Warm-start** from a 3-class NLI checkpoint (e.g. `cross-encoder/nli-deberta-v3-base`). The leftover train pool is dominated by Content / Numeric / Negation / Perspective; rare primitives (Emotion / Relation / Causal / Factual) are concentrated in the balanced test set, so the encoder needs to bring its own knowledge of those phenomena from pre-training.

3. **Class balance:** roughly 1:5 positive:negative. Either pass `class_weight={0: 1.0, 1: 4.0}` or oversample positives. Don't optimize plain accuracy.

4. **Group-aware CV (optional):** for 5-fold CV inside the train split, use `GroupKFold(n_splits=5).split(X, y, groups=train_df["base_doc_id"])`.

5. **Reproducibility:** every randomness here is seeded with `SEED = 42`.
